In [76]:
import json
import math
import pandas as pd
from pathlib import Path
import re
import statistics
from tqdm import tqdm

from modules import plotting

In [77]:
wd = Path(".brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding")
grids = wd.glob("gridsearch_*/")

slurmoutToExp = {}
for grid in tqdm(grids):
    slurmoutToExp[grid.name] = {}
    experiments = list(grid.glob("wgEncode*/"))
    for slurmerrf in (grid / "slurmout").glob("*.err"):
        exp = None
        with open(slurmerrf, "r") as f:
            for line in f:
                m = re.search(r"wgEncode.+\.narrowPeak", line)
                if m:
                    exp = m.group(0)
                    break

        assert exp is not None, f"Could not find experiment name in {slurmerrf}"
        slurmoutToExp[grid.name][exp] = slurmerrf

print(slurmoutToExp)

33it [00:37,  1.14s/it]

{'gridsearch_00008': {'wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak': PosixPath('.brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00008/slurmout/5566978_11_gridsearch_00008.err'), 'wgEncodeAwgTfbsSydhK562Atf3UniPk.narrowPeak': PosixPath('.brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00008/slurmout/5566978_0_gridsearch_00008.err'), 'wgEncodeAwgTfbsSydhK562Rfx5IggrabUniPk.narrowPeak': PosixPath('.brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00008/slurmout/5566978_28_gridsearch_00008.err'), 'wgEncodeAwgTfbsSydhK562Gata2UcdUniPk.narrowPeak': PosixPath('.brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00008/slurmout/5566978_17_gridsearch_00008.err'), 'wgEncodeAwgTfbsHaibK562Nr2f2sc271940V0422111UniPk.narrowPeak': PosixPath('.brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00008/slurmout/5566978_26_gridsearch_00008.err'), 'wgEncodeAwgTfbsSydhK562Usf2IggrabUniPk.narr

In [78]:
# wd = Path(".brain_mnt/runs/20250425_gridsearch_STREME_vs_ProfileFinding_pinky")
# grids = wd.glob("gridsearch_*/")
ngrids = len(list(grids))
runs = []
for grid in tqdm(sorted(list(wd.glob("gridsearch_*/")))):
    # print(f"Processing {grid.name}: {list(grid.glob('wgEncode*/'))}")
    nlogs = len(list((grid / "slurmout").glob("*"))) / 2
    experiments = list(grid.glob("wgEncode*/"))
    # print(f"Experiments: {experiments}")
    nexp = len(experiments)
    if nexp != nlogs:
        print(f"Warning: {grid.name} has {nexp} experiments but {nlogs}*2 logs")
    success = []
    failed = []
    ooms = []
    for exp in experiments:
        # print(f"Processing {exp.name}")
        if (exp / "tomtom" / "tomtom.tsv").exists():
            success.append(exp)
        else:
            failed.append(exp)
            # slurmerrf = slurmoutToExp[grid.name][exp.name]
            slurmerrf = slurmoutToExp[grid.name].get(exp.name, None)
            if slurmerrf is None:
                print(f"Warning: {grid.name} has no slurmout for {exp.name}")
                continue
            with open(slurmerrf, "r") as f:
                for line in f:
                    m = re.search(r"slurmstepd-node\d+: error: Detected (\d+) oom-kill event", line)
                    if m:
                        ooms.append(m.group(1))
                        break

    runs.append(
        {
            "grid": grid.name,
            "nexp": nexp,
            "success": len(success),
            "failed": len(failed),
            "ooms": len(ooms),
            "success_rate": len(success) / nexp if nexp > 0 else None,
        }
    )

print(f"Number of grids: {ngrids}")
print(f"Number of experiments: {sum([r['nexp'] for r in runs])}")
print(f"Number of successful experiments: {sum([r['success'] for r in runs])}")
print(f"Number of failed experiments: {sum([r['failed'] for r in runs])}")
print(f"Number of OOMs: {sum([r['ooms'] for r in runs])}")
print(f"Success rate: {sum([r['success'] for r in runs]) / sum([r['nexp'] for r in runs])}")

pd.DataFrame(runs)

100%|██████████| 33/33 [00:10<00:00,  3.05it/s]

Number of grids: 0
Number of experiments: 1320
Number of successful experiments: 1213
Number of failed experiments: 107
Number of OOMs: 97
Success rate: 0.918939393939394


,grid,nexp,success,failed,ooms,success_rate
0,gridsearch_00000,40,40,0,0,1.000
1,gridsearch_00001,40,40,0,0,1.000
2,gridsearch_00002,40,25,15,5,0.625
3,gridsearch_00003,40,40,0,0,1.000
4,gridsearch_00004,40,40,0,0,1.000
5,gridsearch_00005,40,40,0,0,1.000
6,gridsearch_00006,40,38,2,2,0.950
7,gridsearch_00007,40,38,2,2,0.950
8,gridsearch_00008,40,35,5,5,0.875
9,gridsearch_00009,40,38,2,2,0.950


In [79]:
def _load_tomtom(f) -> float:
    result = pd.read_csv(f, sep="\t", comment="#", header=0)
    if len(result) == 0:
        print(f"[WARNING] >>> Empty tomtom result file {f}, returning -1 as ppval!")
        return -1
    
    pval = result["p-value"].min()
    # negative log10 of p-value
    ppval = -math.log10(pval)
    return ppval

In [80]:
results = {
    "grid": [],
    "experiments": [],
    "runtimes": [],
    "ppvals": [],
}
for grid in tqdm(sorted(list(wd.glob("gridsearch_*/")))):
    experiments = list(grid.glob("wgEncode*/"))
    results["grid"].append(grid.name)
    valid_experiments = []
    runtimes = []
    ppvals = []
    # print(f"Experiments: {experiments}")
    for exp in experiments:
        # print(f"Processing {exp.name}")
        if (exp / "tomtom" / "tomtom.tsv").exists():
            slurmerrf: Path = slurmoutToExp[grid.name][exp.name]
            slourmoutf = slurmerrf.with_suffix(".out")
            runtime = None
            with open(slourmoutf, "r") as f:
                for line in f:
                    m = re.search("Job "+grid.name+r" finished in (\d+) seconds", line)
                    if m:
                        runtime = int(m.group(1))
                        break

            assert runtime is not None, f"Could not find runtime in {slourmoutf}"
            runtimes.append(runtime)
            valid_experiments.append(exp.name)
            tomtomf = exp / "tomtom" / "tomtom.tsv"
            ppval = _load_tomtom(tomtomf)
            ppvals.append(ppval)
        # else:
        #     print(f"Warning: {grid.name} has no tomtom result for {exp.name}")

    results["experiments"].append(valid_experiments)
    results["runtimes"].append(runtimes)
    results["ppvals"].append(ppvals)

  0%|          | 0/33 [00:00<?, ?it/s]

[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00000/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00000/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00000/wgEncodeAwgTfbsSydhK562Znf143IggrabUniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!


  3%|▎         | 1/33 [00:03<01:58,  3.69s/it]

[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00001/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00001/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00001/wgEncodeAwgTfbsHaibK562Zbtb7asc34508V0416101UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!


  6%|▌         | 2/33 [00:06<01:46,  3.43s/it]

...

[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00030/wgEncodeAwgTfbsSydhK562Znf143IggrabUniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00031/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00031/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!


 97%|█████████▋| 32/33 [01:31<00:02,  2.14s/it]

[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00031/wgEncodeAwgTfbsSydhK562Znf143IggrabUniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00032/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!
[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00032/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!


100%|██████████| 33/33 [01:33<00:00,  2.83s/it]

[WARNING] >>> Empty tomtom result file .brain_mnt/runs/20250428_gridsearch_STREME_vs_ProfileFinding/gridsearch_00032/wgEncodeAwgTfbsSydhK562Znf143IggrabUniPk.narrowPeak/tomtom/tomtom.tsv, returning -1 as ppval!


In [81]:
results["experiments"][0]

['wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562Gata2UcdUniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562Elk112771IggrabUniPk.narrowPeak',
 'wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562Atf3V0416101UniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562Usf2IggrabUniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562Ets1V0416101UniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562Bhlhe40nb100IggrabUniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562Nfe2UniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562Gata1UcdUniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562Gata2sc267Pcr1xUniPk.narrowPeak',
 'wgEncodeAwgTfbsSydhK562Atf3UniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562Usf1V0416101UniPk.narrowPeak',
 'wgEncodeAwgTfbsHaibK562E2f6V0416102UniPk.narrowPeak',
 'wg

In [82]:
flat_dict = {
    "grid": [],
    "experiment": [],
    "runtime": [],
    "ppval": [],
}
sum_dict = {
    "grid": [],
    "mean runtime": [],
    "std runtime": [],
    "mean ppval": [],
    "std ppval": [],
}
for i, grid in enumerate(results["grid"]):
    assert len(results["experiments"][i]) == len(results["runtimes"][i])
    assert len(results["experiments"][i]) == len(results["ppvals"][i])
    if len(results["experiments"][i]) == 0:
        print(f"Warning: {grid} has no valid experiments")
        continue
    
    for exp, runtime, ppval in zip(results["experiments"][i], results["runtimes"][i], results["ppvals"][i]):
        flat_dict["grid"].append(grid)
        flat_dict["experiment"].append(exp)
        flat_dict["runtime"].append(runtime)
        flat_dict["ppval"].append(ppval)

    sum_dict["grid"].append(grid)
    sum_dict["mean runtime"].append(statistics.mean(results["runtimes"][i]))
    sum_dict["std runtime"].append(statistics.stdev(results["runtimes"][i]))
    sum_dict["mean ppval"].append(statistics.mean(results["ppvals"][i]))
    sum_dict["std ppval"].append(statistics.stdev(results["ppvals"][i]))

df_flat = pd.DataFrame(flat_dict)
df_sum = pd.DataFrame(sum_dict)

In [83]:
df_sum.sort_values("mean ppval", ascending=False, inplace=True)
df_sum.reset_index(drop=True, inplace=True)
df_sum

,grid,mean runtime,std runtime,mean ppval,std ppval
0,gridsearch_00015,4129.128205,3260.859898,4.886516,2.398714
1,gridsearch_00007,4064.447368,3586.196326,4.666014,2.959800
2,gridsearch_00008,3926.228571,3600.519310,4.214837,2.166985
3,gridsearch_00002,3617.320000,2292.063730,4.086987,2.164988
4,gridsearch_00021,3964.425000,3193.387671,3.927315,2.145406
5,gridsearch_00020,4295.125000,3449.242021,3.923696,2.157344
6,gridsearch_00025,4657.179487,4340.813705,3.913898,2.193587
7,gridsearch_00026,4925.650000,4388.800120,3.907626,2.272405
8,gridsearch_00005,4768.550000,4127.401858,3.894501,2.347384
9,gridsearch_00027,4238.128205,3730.763660,3.886345,2.292470


In [84]:
plotting.boxplotNumColumn(df_flat, column="ppval", byColumn="grid", title="Motif Similarity Score", label="Grid", ylab="-log10(p-value)")

In [86]:
plotting.boxplotNumColumn(df_flat, column="runtime", byColumn="grid", title="Runtime", label="Grid", ylab="runtime (s)")

gridsearch_00015 should be the new baseline for further grid searches. This means mellowmax_alpha = 0.5 works best for 
the current set of parameters.

In [87]:
def get_config(grid: Path) -> dict:
    experiments = list(grid.glob("wgEncode*/"))
    config = None
    for exp in experiments:
        configf = exp / "settings.json"
        assert configf.exists(), f"Config file {configf} does not exist"
        with open(configf, "r") as f:
            _config: dict = json.load(f)
        _config.pop("fasta")
        _config.pop("out")
        if config is not None:
            assert _config == config, f"Config files {configf} and {config} are different"
        else:
            config = _config

    return config

In [88]:
grid_configs = {
    grid.name: get_config(grid) for grid in tqdm(sorted(list(wd.glob("gridsearch_*/"))))
}

  0%|          | 0/33 [00:00<?, ?it/s]

100%|██████████| 33/33 [00:40<00:00,  1.23s/it]


In [90]:
grid_diffs = {
    g: {
        k: v
        for k, v in grid_configs[g].items()
        if grid_configs["gridsearch_00000"][k] != grid_configs[g][k]
    } for g in grid_configs
}
grid_diffs    

{'gridsearch_00000': {},
 'gridsearch_00001': {'U': 50},
 'gridsearch_00002': {'U': 500},
 'gridsearch_00003': {'overlapTilesize': 1},
 'gridsearch_00004': {'overlapTilesize': 10},
 'gridsearch_00005': {'midK': 8},
 'gridsearch_00006': {'midK': 10},
 'gridsearch_00007': {'gamma': 0.5},
 'gridsearch_00008': {'gamma': 0.01},
 'gridsearch_00009': {'gamma': 2.0},
 'gridsearch_00010': {'kld': 0.1},
 'gridsearch_00011': {'kld': 0.001},
 'gridsearch_00012': {'kld': 1.0},
 'gridsearch_00013': {'kld': 0.0},
 'gridsearch_00014': {'mellowmax_alpha': 0.0},
 'gridsearch_00015': {'mellowmax_alpha': 0.5},
 'gridsearch_00016': {'mellowmax_alpha': 2.0},
 'gridsearch_00017': {'mellowmax_alpha': 5.0},
 'gridsearch_00018': {'match_score_factor': 0.6},
 'gridsearch_00019': {'match_score_factor': 0.5},
 'gridsearch_00020': {'match_score_factor': 0.8},
 'gridsearch_00021': {'match_score_factor': 0.9},
 'gridsearch_00022': {'learning_rate': 0.1},
 'gridsearch_00023': {'learning_rate': 1.0},
 'gridsearch_00024

In [91]:
config_df_dict = {
    'parameter': [],
    'grid': [],
    'value': [],
    'mean ppval': [],
    'std ppval': [],
    'delta ppval': [],
    'mean runtime': [],
    'std runtime': [],
    'delta runtime': [],
}
base_ppval = df_sum[df_sum["grid"] == "gridsearch_00000"]["mean ppval"].values[0]
base_ppval_std = df_sum[df_sum["grid"] == "gridsearch_00000"]["std ppval"].values[0]
base_runtime = df_sum[df_sum["grid"] == "gridsearch_00000"]["mean runtime"].values[0]
base_runtime_std = df_sum[df_sum["grid"] == "gridsearch_00000"]["std runtime"].values[0]
current_param = None
for grid, config in grid_diffs.items():
    if len(config) == 0:
        continue
    if grid not in df_sum["grid"].values:
        print(f"Warning: {grid} not in df_sum")
        continue
    assert len(config) == 1, f"Grid {grid} has more than one config parameter"
    param = list(config.keys())[0]
    value = config[param]
    mean_ppval = df_sum[df_sum["grid"] == grid]["mean ppval"].values[0]
    std_ppval = df_sum[df_sum["grid"] == grid]["std ppval"].values[0]
    mean_runtime = df_sum[df_sum["grid"] == grid]["mean runtime"].values[0]
    std_runtime = df_sum[df_sum["grid"] == grid]["std runtime"].values[0]
    if current_param is None or current_param != param:
        config_df_dict["parameter"].append(param)
        config_df_dict["grid"].append("gridsearch_00000")
        config_df_dict["value"].append(grid_configs["gridsearch_00000"][param])
        config_df_dict["mean ppval"].append(base_ppval)
        config_df_dict["std ppval"].append(base_ppval_std)
        config_df_dict["delta ppval"].append(0)
        config_df_dict["mean runtime"].append(base_runtime)
        config_df_dict["std runtime"].append(base_runtime_std)
        config_df_dict["delta runtime"].append(0)
        current_param = param

    config_df_dict["parameter"].append(param)
    config_df_dict["grid"].append(grid)
    config_df_dict["value"].append(value)
    config_df_dict["mean ppval"].append(mean_ppval)
    config_df_dict["std ppval"].append(std_ppval)
    config_df_dict["delta ppval"].append(mean_ppval - base_ppval)
    config_df_dict["mean runtime"].append(mean_runtime)
    config_df_dict["std runtime"].append(std_runtime)
    config_df_dict["delta runtime"].append(mean_runtime - base_runtime)

config_df = pd.DataFrame(config_df_dict)
print(df_sum[df_sum['grid'] == 'gridsearch_00000'])
config_df

                grid  mean runtime  std runtime  mean ppval  std ppval
10  gridsearch_00000        4792.4  4031.454505    3.852606   2.167418


,parameter,grid,value,mean ppval,std ppval,delta ppval,mean runtime,std runtime,delta runtime
0,U,gridsearch_00000,200.000,3.852606,2.167418,0.000000,4792.400000,4031.454505,0.000000
1,U,gridsearch_00001,50.000,3.734047,2.108437,-0.118559,3212.150000,3203.221564,-1580.250000
2,U,gridsearch_00002,500.000,4.086987,2.164988,0.234381,3617.320000,2292.063730,-1175.080000
3,overlapTilesize,gridsearch_00000,6.000,3.852606,2.167418,0.000000,4792.400000,4031.454505,0.000000
4,overlapTilesize,gridsearch_00003,1.000,3.776155,2.331067,-0.076451,4745.700000,4009.973829,-46.700000
5,overlapTilesize,gridsearch_00004,10.000,3.779584,2.223392,-0.073022,4741.275000,4060.250984,-51.125000
6,midK,gridsearch_00000,6.000,3.852606,2.167418,0.000000,4792.400000,4031.454505,0.000000
7,midK,gridsearch_00005,8.000,3.894501,2.347384,0.041895,4768.550000,4127.401858,-23.850000
8,midK,gridsearch_00006,10.000,3.482036,2.268724,-0.370570,4300.631579,3778.821827,-491.768421
9,gamma,gridsearch_00000,1.000,3.852606,2.167418,0.000000,4792.400000,4031.454505,0.000000


In [75]:
pseudo_optimal_config = {
    'U': 200,
    'overlapTilesize': 1,
    'midK': 6,
    'gamma': 0.01,
    'kld': 0.1,
    'mellowmax_alpha': 0.5,
    'match_score_factor': 0.8,
    'learning_rate': 1.0,
    'lr_patience': 15,
    'lr_factor': 0.25,
    'profile_plateau': 5,
    'profile_plateau_dev': 100,
}

for k, v in grid_configs['gridsearch_00000'].items():
    if k not in pseudo_optimal_config:
        pseudo_optimal_config[k] = v

print(pseudo_optimal_config)

with open(wd / "pseudo_optimal_config.json", "w") as f:
    json.dump(pseudo_optimal_config, f, indent=4)

{'U': 200, 'overlapTilesize': 1, 'midK': 6, 'gamma': 0.01, 'kld': 0.1, 'mellowmax_alpha': 0.5, 'match_score_factor': 0.8, 'learning_rate': 1.0, 'lr_patience': 15, 'lr_factor': 0.25, 'profile_plateau': 5, 'profile_plateau_dev': 100, 'mode': 'DNA', 'config': None, 'maxseqs': None, 'no_softmasking': False, 'do_not_train': False, 'rand_seed': 42, 'tile_size': 100, 'tiles_per_X': 1, 'batch_size': 1, 'prefetch': 3, 'n_best_profiles': 5, 'enforceU': False, 'minU': 10, 'minOcc': 8, 'k': 12, 's': 0, 'l2': 0.0, 'rho': 0.0, 'sigma': 1.0, 'phylo_t': 0.0}
